# Testing
## Bevölkerung nach Geburtsort und Geburtsland
Datum: 21.01.2022

### Settings

#### Importiere die notwendigen Packages

In [1]:
#pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime plotly

In [2]:
import pandas as pd
import pivottablejs
from pivottablejs import pivot_ui
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
from datetime import datetime
import geopandas as gpd
import folium 
import requests

import datetime
import time

In [3]:
import plotly.express as px


Definiere Settings. Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen), 

In [4]:
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 15)

In [5]:
now = datetime.date.today()
date_today = now.strftime("%Y-%m-%d")

#### Datenimport

**Dataset auf INTEG Datenkatalog**: https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_bestand_jahr_gemeinde_land_geburtsort_od3380

**Dataset auf PROD Datenkatalog**: https://data.stadt-zuerich.ch/dataset/bev_bestand_jahr_gemeinde_land_geburtsort_od3380


In [6]:
#Die Datasets sind nur zum Testen auf INT-DWH-Dropzone. Wenn der Test vorbei ist, sind sie auf PROD. 
# Über den Status kann man einfach switchen

status = "prod";
print(status)

prod


In [7]:
# Filepath
if status == "prod":
    fp = r"//szh.loc/ssz/applikationen/OGD_Dropzone/DWH/bev_bestand_jahr_gemeinde_land_geburtsort_od3380/BEV338OD3380.csv"
else:
    fp = r"//szh.loc/ssz/applikationen/OGD_Dropzone/INT_DWH/bev_bestand_jahr_gemeinde_land_geburtsort_od3380/BEV338OD3380.csv"
#download from INT-Datacatalog:
# https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bfs_bevreg_todesursache-aggregiert_alter_geschlecht_seit1995_od3120/download/BEV312OD3120.csv
print(fp)

//szh.loc/ssz/applikationen/OGD_Dropzone/DWH/bev_bestand_jahr_gemeinde_land_geburtsort_od3380/BEV338OD3380.csv


In [8]:
# Read the data
data2betested = pd.read_csv(
    fp, 
    sep=',',
    parse_dates=True,
    #date_parser=lambda x: pd.to_datetime(x, format="%Y"), 
    #index_col="date_time",
    low_memory=False    
    )

#### Basic Tests

In der Folge ein paar erste Tests:

In [9]:
#type(data2betested)

 - 1) Zeige eine kurze Vorschau der importierten Daten
 - 2) Weise die Datentypen aus
 - 3) Zeige die Shape (Umfang) des Datensatzes an

In [10]:
data2betested.head(1)

,StichtagDatJahr,xkoord,ykoord,GebOrtSort,GebOrtLang,GebLandSort,GebLandLang,AnzBestWir
0,2020,2679530,1236588,1,Aeugst am Albis,8100,Schweiz,6


In [11]:
data2betested.dtypes

StichtagDatJahr      int64
xkoord             float64
ykoord             float64
GebOrtSort           int64
GebOrtLang          object
GebLandSort          int64
GebLandLang         object
AnzBestWir           int64
dtype: object

In [12]:
data2betested.shape

(1696, 8)

Beschreibe einzelne Attribute

In [13]:
data2betested.describe()

,StichtagDatJahr,xkoord,ykoord,GebOrtSort,GebLandSort,AnzBestWir
count,1696,1487,1487,1696,1696,1696
mean,2020,2654437,1214251,3707,8136,256
std,0,61281,47532,2979,106,3301
min,2020,2495123,1076471,1,8100,0
25%,2020,2611483,1183438,1058,8100,2
50%,2020,2655114,1226016,3312,8100,5
75%,2020,2698298,1252343,5173,8100,18
max,2020,2826234,1291100,9999,8995,126647


#### In diesem Datensatz insbesondere Missings bei Koordinaten?

Wie viele Nullwerte gibt es im Datensatz?

In [14]:
data2betested.isnull().sum()

StichtagDatJahr      0
xkoord             209
ykoord             209
GebOrtSort           0
GebOrtLang           0
GebLandSort          0
GebLandLang          0
AnzBestWir           0
dtype: int64

Viele Missings stammen daher, dass wir keine Koordinaten zu Ländern haben. Aus der Schweiz sollte es aber keine geben...

In [15]:
ch_missings = data2betested.query('GebLandLang=="Schweiz"')
ch_missings[ch_missings['xkoord'].isna()]

,StichtagDatJahr,xkoord,ykoord,GebOrtSort,GebOrtLang,GebLandSort,GebLandLang,AnzBestWir
629,2020,NaN,NaN,2430,Welschenrohr-Gänsbrunnen,8100,Schweiz,15
911,2020,NaN,NaN,3715,Muntogna da Schons,8100,Schweiz,1
1306,2020,NaN,NaN,5399,Verzasca,8100,Schweiz,1


**Achtung**

Wenn es hier Missings hat, dann bedeutet das, dass im SASA-Job wohl nicht die aktuellsten Koordinatenwerte übernommen wurden. Diese müssen jedes Jahr ca. Ende Januar von der Swisstopo aufbereitet werden. Bei uns macht das Mario, für die Gemeindegrenzen.


### Zwischenschritt optional - siehe dazu das Verzeichnis «update_ogd» > ch_gemeinde_centroide

Wechsle dazu in das update_ogd Verzeichnis

### Weiter mit Plausi

Wandle die Spalte Jahr von Int zu **Datetime** um. Dies wird in den meisten Grafiken so benötigt.

In [17]:
data2betested['StichtagDatJahr']=pd.to_datetime(data2betested['StichtagDatJahr'], format='%Y')

#TO DO --> setze das Datum auf den 31.12. der Jahres.
#test=data2betested[['StichtagDatJahr']].date().replace(month=12, day=31)

data2betested

,StichtagDatJahr,xkoord,ykoord,GebOrtSort,GebOrtLang,GebLandSort,GebLandLang,AnzBestWir
0,2020-01-01,2679530,1236588,1,Aeugst am Albis,8100,Schweiz,6
1,2020-01-01,2676539,1237245,2,Affoltern am Albis,8100,Schweiz,1290
2,2020-01-01,2677695,1241150,3,Bonstetten,8100,Schweiz,5
3,2020-01-01,2683128,1233187,4,Hausen am Albis,8100,Schweiz,18
4,2020-01-01,2676560,1239448,5,Hedingen,8100,Schweiz,8
...,...,...,...,...,...,...,...,...
1691,2020-01-01,NaN,NaN,9999,Ausland,8612,Samoa,1
1692,2020-01-01,NaN,NaN,9999,Ausland,8614,Salomoninseln,2
1693,2020-01-01,NaN,NaN,9999,Ausland,8615,Tuvalu,1
1694,2020-01-01,NaN,NaN,9999,Ausland,8621,Amerikanisch-Samoa,1


Bereite die Daten für die Grafik unten vor

In [18]:
Jahre = data2betested.groupby("StichtagDatJahr")
len(Jahre)

1

In [19]:
for key, group in Jahre:
    print(key, len(group))

2020-01-01 00:00:00 1696


In [20]:
myJahr = "2020-01-01"
aktuellesJahr=Jahre.get_group(myJahr)   
aktuellesJahr.head(3)

,StichtagDatJahr,xkoord,ykoord,GebOrtSort,GebOrtLang,GebLandSort,GebLandLang,AnzBestWir
0,2020-01-01,2679530,1236588,1,Aeugst am Albis,8100,Schweiz,6
1,2020-01-01,2676539,1237245,2,Affoltern am Albis,8100,Schweiz,1290
2,2020-01-01,2677695,1241150,3,Bonstetten,8100,Schweiz,5


**Selektionen und Aggregationen**

Check this [Doku](https://www.shanelynn.ie/summarising-aggregation-and-grouping-data-in-python-pandas/)

In [ ]:
test = data2betested.groupby(
   ['GebOrtSort', 'GebOrtLang', 'StichtagDatJahr']
).agg(
    {
         'AnzBestWir':(sum),    # Sum --> (sum)
         'FlaecheS': 'max',  # get the first date per group
        'StichtagDatJahr': 'first'
    }
)
test.head()
    

**Einfache Visualisierungen zur Plausi**

Exploriere die Daten mit Pivottable.JS

In [22]:
from pivottablejs import pivot_ui

pivot_ui(data2betested)

Liniendiagramm 
[Link zur Doku](https://altair-viz.github.io/gallery/multiline_highlight.html)

In [23]:
mySelection = data2betested[(data2betested.StichtagDatJahr>"2011-01-01") & (data2betested.RaumKategorie=='Stadtquartier') & (data2betested.RaumSort<70)]

mySelection.head()



AttributeError: 'DataFrame' object has no attribute 'RaumKategorie'

In [ ]:
#mySelection = aktuellesJahr[["StatZoneSort","AnzBestWir"]]
#mySelection

In [ ]:
# df_statzonen = df_statzonen[(df_statzonen.knr>0) & (df_statzonen.qname!="Hirslanden")]
#aktuellesJahr[[aktuellesJahr.KreisSort =7]])
highlight = alt.selection(type='single', on='mouseover',
                          fields=['RaumLang'], nearest=True)
#x='date:StichtagDatJahr',
base = alt.Chart(mySelection).encode(
    x='StichtagDatJahr:N',
    y='DichteS',
    color='RaumLang'
)

points = base.mark_circle().encode(
    opacity=alt.value(0)
).add_selection(
    highlight
).properties(
    width=700 , height=300
)
lines = base.mark_line().encode(
    size=alt.condition(~highlight, alt.value(1.5), alt.value(4))
).interactive()

points + lines

In [ ]:
alt.Chart(data2betested).mark_circle(size=60).encode(
    x='RaumLang', #QuarLang
    y='StichtagDatJahr',
    #color='AnzNat',
    #color=alt.Color('AnzNat', scale=alt.Scale(scheme='dark2')),
    color=alt.Color('DichteS', scale=alt.Scale(scheme='reds')),
    tooltip=['RaumLang','StichtagDatJahr','DichteS']
).interactive() # this makes the axes interactive: now you can zoom & pan




Test: Choroplethenkarte

**Test: Choroplethenkarte**
Importiere die Geodaten als GeoJSON

In [ ]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
# Filepath statzonen
input_stadtquartiere = r"//szh.loc/ssz/data/GIS/Daten/Vektor/_aktuell/stzh.adm_statistische_quartiere_map.json"
df_stadtquartiere = gpd.read_file(input_stadtquartiere)
df_stadtquartiere.head(2)
#df_stadtkreise.dtypes


Joine die importierten statistischen Daten des aktuellen Jahres zum Geodatensatz:

*Siehe dazu Doku zu [Geopandas](https://geopandas.org/docs/user_guide/mergingdata.html)*

In [ ]:
aktuellesJahr = data2betested[(data2betested.StichtagDatJahr=="2020-01-01") & (data2betested.RaumKategorie=='Stadtquartier') & (data2betested.RaumSort<500)]

aktuellesJahr.head(2)

In [ ]:
stats_agg_quartiere=aktuellesJahr
#stats_agg_quartiere=aktuellesJahr.groupby(['StichtagDatJahr', 'RaumLang', 'AnzBestWir'])['AnzNat'].sum()

#stats_agg_quartiere.head()

In [ ]:
# Rename attribute stznr to StatZoneSort, so the IDs have the same name

stadtquartiere_shapes = df_stadtquartiere[['geometry', 'qnr', 'qname', 'knr']].rename(columns={'qnr': 'RaumSort'})
#stadtquartiere_shapes = df_stadtquartiere[['geometry', 'qnr', 'qname', 'knr']].rename(columns={'qnr': 'QuarSort'})

# Merge with `merge` method on shared variable (stznr und StatZoneSort):
stadtquartiere_shapes_joined = stadtquartiere_shapes.merge(stats_agg_quartiere, on='RaumSort')

stadtquartiere_shapes_joined.head(3)


Importiere die Gemeindegrenzen der Schweiz und exportiere die Centroide dazu

In [ ]:
# Plot by StatZonen

stadtquartiere_shapes_joined.plot(column='DichteS', cmap='viridis', scheme='natural_breaks', legend=True )
#scheme werte: https://matplotlib.org/2.0.2/users/colormaps.html

stadtquartiere_shapes_joined.plot(column='DichteS', cmap='inferno', scheme='quantiles', legend=True )
#scheme werte: https://matplotlib.org/2.0.2/users/colormaps.html
stadtquartiere_shapes_joined.plot(column='DichteS', cmap='magma', scheme='fisherjenks', legend=True )
#scheme werte: https://matplotlib.org/2.0.2/users/colormaps.html

Noch zu prüfen, folgendes schöne Beispiel: https://docs.bokeh.org/en/latest/docs/user_guide/interaction/legends.html

Konkrete Beispiele von Bookeh in Jupyter NB: https://docs.bokeh.org/en/latest/docs/user_guide/jupyter.html

**Sharepoint Status anpassen!**

Record auf Sharepoint in Bearbeitungsmodus öffnen: **[Link](http://kollaboration.intranet.stzh.ch/orga/ssz-produkte/Lists/SASA_Outputs/EditForm.aspx?ID=831&Source=%2Forga%2Fssz%2Dprodukte%2FLists%2FSASA%5FOutputs)**